# Step 06.2 — One-Hot vs Learned Embedding

6.1 已经把字符变成 token ID，例如：

```text
a -> 0
b -> 1
c -> 2
d -> 3
```

但这里有一个非常危险的误解：**token ID 是类别编号，不是连续数值特征。**

这一节完整解决 5 个问题：

1. 为什么不能把 token ID 直接当数值输入模型；
2. One-Hot 怎样表达“只有身份，没有大小”；
3. Embedding table 怎样把离散 ID 变成可学习向量；
4. 为什么 embedding 可以形成有用几何结构，但不保证出现人类直觉聚类；
5. 最后把 shape 接到 MiniGPT 的 `(B,T,C)`。

## 1. Token ID 为什么不能直接当普通数值？——ID 会制造假的“大小”和“距离”

假设 vocabulary：

```text
cat -> 0
dog -> 1
sun -> 2
car -> 3
```

如果直接把这些 ID 当实数输入神经网络，就等于偷偷告诉模型：

```text
dog - cat = 1
sun - dog = 1
car - cat = 3
```

这会产生一个根本不存在的几何关系：好像 `cat` 与 `dog` 比 `cat` 与 `car` 更接近，只因为编号碰巧如此。

但 tokenizer 的编号顺序本来可以任意换：

```text
cat -> 3
dog -> 0
sun -> 1
car -> 2
```

语义完全没变，数值距离却全变了。

所以：

> **Token ID 应该被理解成“查表地址 / category label”，而不是可做加减乘除的语义数值。**

In [ ]:
tokens = ["cat", "dog", "sun", "car"]
ids = [0, 1, 2, 3]

for token, token_id in zip(tokens, ids):
    print(f"{token:>3} -> {token_id}")

print("\nFake numeric distances if IDs are treated as values:")
print("|dog-cat| =", abs(1 - 0))
print("|car-cat| =", abs(3 - 0))

### 用图把这个错误直觉画出来

这里第一次正式引入一个很实用的科研绘图套路：

```text
1. plt.figure(...)   -> 新建画布
2. plt.scatter(...)  -> 画点
3. plt.annotate(...) -> 给点加文字标签
4. plt.xlabel/title  -> 补坐标含义
5. plt.show()        -> 显示图
```

以后画 loss curve、embedding、benchmark 都会反复使用这几个基本动作。

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 2.2))
plt.scatter(ids, [0] * len(ids))

for token, token_id in zip(tokens, ids):
    plt.annotate(token, (token_id, 0), xytext=(0, 10), textcoords="offset points", ha="center")

plt.yticks([])
plt.xlabel("token ID")
plt.title("Wrong mental model: token IDs placed on a numeric line")
plt.show()

这张图的目的不是说 tokenizer 真有这条数轴，而是恰恰相反：

> **如果你直接把 ID 当连续数值，模型就会被迫看到这样一条人为制造的数轴。**

## 2. One-Hot：只表达“是谁”，不表达“谁比谁大”

如果 vocabulary 有 4 个 token，我们可以给每个 token 一个 4 维向量：

```text
token 0 -> [1,0,0,0]
token 1 -> [0,1,0,0]
token 2 -> [0,0,1,0]
token 3 -> [0,0,0,1]
```

这叫 **One-Hot Encoding**。

它有两个重要性质：

### A. 没有大小顺序

`token 3` 不会因为 ID=3 就数值更大；每个 token 都只是不同坐标轴上的一个 1。

### B. 所有不同 token 在这种表示下完全对称

例如：

```text
[1,0,0,0] · [0,1,0,0] = 0
```

任意两个不同 One-Hot 向量的 dot product 都是 0。

它解决了“假的数字大小”问题，但它**还没有学习任何语义相似性**。

In [ ]:
import mlx.core as mx

V = 4
one_hot_table = mx.eye(V)

print(one_hot_table)
print("shape:", one_hot_table.shape)

cat = one_hot_table[0]
dog = one_hot_table[1]
car = one_hot_table[3]

print("cat·dog =", mx.sum(cat * dog).item())
print("cat·car =", mx.sum(cat * car).item())

One-Hot 还有一个明显成本：

如果：

```text
vocab_size = V = 50,000
```

那么每个 token 的 One-Hot 向量长度就是 50,000，而且绝大多数位置都是 0。

它在概念上非常干净，但作为神经网络内部表示非常浪费。

## 3. Learned Embedding：把“身份”变成可训练的稠密向量

现在准备一个矩阵：

```text
E.shape = (V, C)
```

其中：

- `V` = vocab size
- `C` = embedding dimension / model width

每一行对应一个 token 的向量：

```text
E =
[ token 0 的 C 维向量 ]
[ token 1 的 C 维向量 ]
[ token 2 的 C 维向量 ]
...
```

因此 token ID 的真正用途变成：

```text
token_id = 2
↓
取 E 的第 2 行
↓
得到一个 C 维向量
```

这就是 embedding lookup 的核心。

In [ ]:
# 手工造一个很小的 embedding table：V=4, C=2
E = mx.array([
    [ 1.0,  0.2],   # token 0
    [ 0.8,  0.3],   # token 1
    [-0.5,  1.0],   # token 2
    [-0.7,  0.9],   # token 3
])

token_id = 1
embedding = E[token_id]

print("E shape    :", E.shape)
print("token_id   :", token_id)
print("embedding  :", embedding)
print("embed shape:", embedding.shape)

### One-Hot 与 Embedding lookup 的数学联系

如果 token 1 的 one-hot 是：

```text
[0,1,0,0]
```

那么：

```text
[0,1,0,0] @ E
```

刚好就是把 `E` 的第 1 行选出来。

所以从数学上：

```text
one_hot(token_id) @ E
```

等价于：

```text
E[token_id]
```

但真实实现没有必要真的构造巨大 One-Hot；直接按 ID 查行更省内存、更高效。

In [ ]:
one_hot = one_hot_table[token_id]

via_matmul = one_hot @ E
via_lookup = E[token_id]

mx.eval(via_matmul, via_lookup)

print("one-hot @ E:", via_matmul)
print("E[id]      :", via_lookup)
print("same       :", bool(mx.allclose(via_matmul, via_lookup)))

### 一个必须纠正的误区：`C` 不需要大于等于 `V`

Embedding dimension 和 vocabulary size 是两个不同概念。

```text
V = 有多少种 token
C = 每个 token 用多少个连续特征表示
```

它们没有“必须 `C ≥ V` 才能区分 token”的要求。

例如现代模型完全可能：

```text
V = 50,000
C = 768
```

50,000 个 token 都可以各自拥有一个 768 维向量；并不要求 50,000 个向量两两正交。

反过来，我们的字符级教学模型可能只有约 65 个 token，却选择 `C=128`。这也完全合法。

## 4. Embedding 的几何：它为什么比 One-Hot 更有表达力？

One-Hot 中所有不同 token 都处于完全对称的位置：彼此 dot product 都是 0，没有谁天然更相似。

Learned embedding 不一样。`E` 是 parameter，会被训练修改。

训练可以让模型为了 next-token prediction，把某些 token 的向量调到对任务更有用的位置。

例如下面这张二维图只是**手工构造的教学示意**，不是说真实模型一定会得到这种漂亮聚类。

In [ ]:
toy_tokens = ["cat", "dog", "sun", "car"]
toy_E = [
    [0.9, 0.8],
    [1.1, 0.9],
    [-0.9, 1.0],
    [-0.8, -0.7],
]

xs = [p[0] for p in toy_E]
ys = [p[1] for p in toy_E]

plt.figure(figsize=(6, 5))
plt.scatter(xs, ys)

for token, px, py in zip(toy_tokens, xs, ys):
    plt.annotate(token, (px, py), xytext=(6, 6), textcoords="offset points")

plt.axhline(0, linewidth=0.8)
plt.axvline(0, linewidth=0.8)
plt.xlabel("embedding dimension 0")
plt.ylabel("embedding dimension 1")
plt.title("Toy 2D embedding geometry (illustration only)")
plt.show()

### 顺手学一个画 embedding 图的小技巧

`scatter` 负责画坐标点；`annotate` 负责把 token 名字贴到点旁边。

真实 embedding 常有几百维，不能直接画。通常需要先用 PCA、t-SNE、UMAP 等方法降到 2D/3D，再画散点图。

但要非常谨慎：

> 降维图是分析工具，不是语义证明。

而且 learned embedding **不保证**自动形成“元音一堆、标点一堆”这种人类容易解释的簇；它真正优化的是训练目标。

## 5. 接到 MiniGPT：从 `(B,T)` token IDs 到 `(B,T,C)` hidden vectors

现在把前面全部合起来。

一句文本经过 tokenizer：

```text
"abcd"
↓
[0,1,2,3]
```

如果 embedding table：

```text
E.shape = (V,C)
```

对整个 ID 序列一次查表：

```text
ids.shape        = (T,)
E[ids].shape     = (T,C)
```

如果加入 Batch：

```text
ids.shape        = (B,T)
embedding output = (B,T,C)
```

这就是我们前面一直提前使用的 `(B,T,C)` 第一次真正从文本中产生出来。

In [ ]:
# V=4, C=2 的小例子
ids_1d = mx.array([0, 1, 2, 3])
vectors_1d = E[ids_1d]

ids_batch = mx.array([
    [0, 1, 2],
    [3, 2, 1],
])
vectors_batch = E[ids_batch]

print("ids_1d       :", ids_1d.shape)
print("vectors_1d   :", vectors_1d.shape)
print("ids_batch    :", ids_batch.shape)
print("vectors_batch:", vectors_batch.shape)

这里还有一个参数量公式：

```text
embedding parameters = V * C
```

例如字符级模型：

```text
V = 65
C = 128
```

则 token embedding table 有：

```text
65 * 128 = 8,320 parameters
```

这些数值一开始随机初始化，后面会通过 gradient descent 被训练。

### 6.2 最终结论

- token ID 是类别编号，不是连续数值；
- 直接把 ID 当数值会制造假的大小与距离；
- One-Hot 消除了这种顺序，但维度等于 `V`，稀疏且没有学习到语义几何；
- learned embedding 使用参数矩阵 `E.shape=(V,C)`；
- `one_hot @ E` 在数学上等价于 `E[token_id]`，实际实现通常直接 lookup；
- `C` 与 `V` 没有 `C≥V` 的要求；
- embedding geometry 是训练目标塑造出来的，不保证形成整齐的人类可解释簇；
- token IDs `(B,T)` 查表后得到 `(B,T,C)`；
- 下一节 6.3：如何从一段 token 序列同时构造 input 与 target，让每个位置都变成一个 next-token prediction 训练题。